In [ ]:
# ============================================================
# CELL 1 — SETUP AND EXTRACT MODEL ARTIFACTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil, gc, joblib, platform, psutil
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/CICIDS2018"

ZIP_AE   = os.path.join(BASE_DIR, "model_AE_MLP (1).zip")
ZIP_MLP  = os.path.join(BASE_DIR, "model_MLP (1).zip")
ZIP_LGBM = os.path.join(BASE_DIR, "model_LightGBM (1).zip")

DATASET_SPLIT = os.path.join(BASE_DIR, "dataset_split.joblib")

EXTRACT_BASE = "/content/model_benchmark_reviewer"
AE_DIR   = os.path.join(EXTRACT_BASE, "ae_mlp")
MLP_DIR  = os.path.join(EXTRACT_BASE, "mlp")
LGBM_DIR = os.path.join(EXTRACT_BASE, "lgbm")

OUT_DIR = os.path.join(BASE_DIR, "benchmark_reviewer_results")
os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(EXTRACT_BASE):
    shutil.rmtree(EXTRACT_BASE)

os.makedirs(AE_DIR, exist_ok=True)
os.makedirs(MLP_DIR, exist_ok=True)
os.makedirs(LGBM_DIR, exist_ok=True)

def extract_zip(zip_path, out_dir):
    print("=" * 80)
    print("Extracting:", zip_path)
    print("To        :", out_dir)

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"ZIP tidak ditemukan: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    print("Extracted files:")
    for root, dirs, files in os.walk(out_dir):
        for f in files:
            print(" -", os.path.join(root, f))

extract_zip(ZIP_AE, AE_DIR)
extract_zip(ZIP_MLP, MLP_DIR)
extract_zip(ZIP_LGBM, LGBM_DIR)

print("\nEnvironment:")
print("Python       :", platform.python_version())
print("CPU count    :", psutil.cpu_count(logical=True))
print("Total RAM GB :", round(psutil.virtual_memory().total / (1024**3), 2))

print("\nCell 1 selesai.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting: /content/drive/MyDrive/CICIDS2018/model_AE_MLP (1).zip
To        : /content/model_benchmark_reviewer/ae_mlp
Extracted files:
 - /content/model_benchmark_reviewer/ae_mlp/ae_mlp_final.h5
 - /content/model_benchmark_reviewer/ae_mlp/ae_model_final.h5
 - /content/model_benchmark_reviewer/ae_mlp/threshold_ae_mlp.pkl
 - /content/model_benchmark_reviewer/ae_mlp/results_ae.joblib
 - /content/model_benchmark_reviewer/ae_mlp/ae_config.joblib
 - /content/model_benchmark_reviewer/ae_mlp/ae_encoder_final.h5
 - /content/model_benchmark_reviewer/ae_mlp/feature_names.pkl
 - /content/model_benchmark_reviewer/ae_mlp/scaler.pkl
Extracting: /content/drive/MyDrive/CICIDS2018/model_MLP (1).zip
To        : /content/model_benchmark_reviewer/mlp
Extracted files:
 - /content/model_benchmark_reviewer/mlp/threshold_mlp.pkl
 - /content/model_benchmark_reviewer/mlp/results_mlp.

In [ ]:
# ============================================================
# CELL 2 — CREATE STANDALONE X_TEST FILE
# ============================================================

import os, gc, joblib
import numpy as np

X_TEST_NPY = os.path.join(EXTRACT_BASE, "X_test_full.npy")

print("Loading dataset_split.joblib:")
print(DATASET_SPLIT)

data = joblib.load(DATASET_SPLIT)

print("\nKeys:")
print(list(data.keys()))

X_test = data["X_test"].astype("float32")

print("\nX_test shape :", X_test.shape)
print("X_test dtype :", X_test.dtype)
print("X_test size  :", round(X_test.nbytes / (1024**2), 2), "MiB")

np.save(X_TEST_NPY, X_test)

print("\nSaved standalone X_test to:")
print(X_TEST_NPY)

del data, X_test
gc.collect()

X_check = np.load(X_TEST_NPY, mmap_mode="r")
print("\nVerification:")
print("Loaded shape:", X_check.shape)
print("Loaded dtype:", X_check.dtype)
print("Loaded size :", round(X_check.nbytes / (1024**2), 2), "MiB")

del X_check
gc.collect()

print("\nCell 2 selesai.")

Loading dataset_split.joblib:
/content/drive/MyDrive/CICIDS2018/dataset_split.joblib

Keys:
['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test', 'label_train', 'label_val', 'label_test', 'features', 'n_features', 'class_weights', 'scale_pos_weight']

X_test shape : (1759848, 38)
X_test dtype : float32
X_test size  : 255.1 MiB

Saved standalone X_test to:
/content/model_benchmark_reviewer/X_test_full.npy

Verification:
Loaded shape: (1759848, 38)
Loaded dtype: float32
Loaded size : 255.1 MiB

Cell 2 selesai.


In [ ]:
# ============================================================
# CELL 3 — CREATE REVIEWER-COMPLIANT BENCHMARK SCRIPT
# ============================================================

import os

SCRIPT_PATH = os.path.join(EXTRACT_BASE, "benchmark_reviewer_single_model.py")

script_code = r'''
import os, gc, time, argparse, joblib, psutil, platform, json, threading, warnings
import numpy as np
import pandas as pd

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

BASE = "/content/model_benchmark_reviewer"
X_TEST_NPY = os.path.join(BASE, "X_test_full.npy")

AE_DIR   = os.path.join(BASE, "ae_mlp")
MLP_DIR  = os.path.join(BASE, "mlp")
LGBM_DIR = os.path.join(BASE, "lgbm")

OUT_DIR = "/content/drive/MyDrive/CICIDS2018/benchmark_reviewer_results"
os.makedirs(OUT_DIR, exist_ok=True)

CHUNK_SIZE = 10000
BATCH_SIZE = 4096

process = psutil.Process(os.getpid())

def rss_mb():
    return process.memory_info().rss / (1024 ** 2)

def cpu_percent_from_process_time(cpu_start, cpu_end, elapsed):
    cpu_time = (cpu_end.user + cpu_end.system) - (cpu_start.user + cpu_start.system)
    n_cpu = psutil.cpu_count(logical=True)
    return (cpu_time / elapsed) / n_cpu * 100 if elapsed > 0 else 0

class MemoryMonitor:
    def __init__(self, interval=0.02):
        self.interval = interval
        self.stop_event = threading.Event()
        self.peak_mb = rss_mb()
        self.thread = threading.Thread(target=self._monitor)

    def _monitor(self):
        while not self.stop_event.is_set():
            current = rss_mb()
            if current > self.peak_mb:
                self.peak_mb = current
            time.sleep(self.interval)

    def start(self):
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        self.thread.join()
        self.peak_mb = max(self.peak_mb, rss_mb())

def find_file(base_dir, filename):
    for root, dirs, files in os.walk(base_dir):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"{filename} not found in {base_dir}")

def file_size_mb(path):
    return os.path.getsize(path) / 1_000_000

def get_serialized_model_size(model_name):
    if model_name == "AE-MLP":
        files = [
            find_file(AE_DIR, "ae_model_final.h5"),
            find_file(AE_DIR, "ae_encoder_final.h5"),
            find_file(AE_DIR, "ae_mlp_final.h5"),
        ]
        return sum(file_size_mb(f) for f in files)

    if model_name == "MLP":
        return file_size_mb(find_file(MLP_DIR, "mlp_model_final.h5"))

    if model_name == "LightGBM":
        return file_size_mb(find_file(LGBM_DIR, "lgbm_model_final.joblib"))

    raise ValueError("Unknown model name")

def load_x_test_memmap():
    print("Loading X_test as memory-mapped array...")
    X = np.load(X_TEST_NPY, mmap_mode="r")
    print(f"X_test shape: {X.shape}")
    print(f"X_test dtype: {X.dtype}")
    print(f"X_test size : {X.nbytes / (1024**2):.2f} MiB")
    return X

def iter_chunks(X, chunk_size):
    n = len(X)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        yield start, end, np.asarray(X[start:end], dtype=np.float32)

def benchmark(model_name, predict_func, X):
    print("=" * 80)
    print(f"Benchmark model : {model_name}")
    print(f"Samples         : {len(X):,}")
    print(f"Mode            : full-test chunked inference, single-model process")
    print(f"Chunk size      : {CHUNK_SIZE:,}")
    print(f"Batch size      : {BATCH_SIZE:,}")
    print("=" * 80)

    rss_after_model_load = rss_mb()

    # Warm-up tidak dihitung ke waktu utama
    print("Warm-up...")
    warmup_chunk = np.asarray(X[:CHUNK_SIZE], dtype=np.float32)
    _ = predict_func(warmup_chunk)
    del warmup_chunk
    gc.collect()

    rss_after_warmup = rss_mb()

    monitor = MemoryMonitor(interval=0.02)

    cpu_start = process.cpu_times()
    start_time = time.perf_counter()
    monitor.start()

    n_processed = 0

    for idx, (start, end, X_chunk) in enumerate(iter_chunks(X, CHUNK_SIZE), start=1):
        _ = predict_func(X_chunk)
        n_processed += len(X_chunk)

        del X_chunk, _
        if idx % 20 == 0:
            print(f"Processed: {n_processed:,} / {len(X):,}")
            gc.collect()

    monitor.stop()
    elapsed = time.perf_counter() - start_time
    cpu_end = process.cpu_times()

    latency_ms = elapsed / n_processed * 1000
    throughput = n_processed / elapsed
    cpu_usage = cpu_percent_from_process_time(cpu_start, cpu_end, elapsed)

    peak_inference_memory = monitor.peak_mb
    model_load_memory_delta = rss_after_model_load
    peak_extra_inference_memory = max(0.0, peak_inference_memory - rss_after_warmup)
    serialized_size_mb = get_serialized_model_size(model_name)

    result = {
        "Model": model_name,
        "Samples": int(n_processed),
        "Total inference time (s)": float(elapsed),
        "Latency per flow (ms)": float(latency_ms),
        "Throughput (flows/s)": float(throughput),
        "CPU usage (%)": float(cpu_usage),

        "Peak Inference Memory RSS (MB)": float(peak_inference_memory),
        "RSS after model load (MB)": float(rss_after_model_load),
        "RSS after warm-up (MB)": float(rss_after_warmup),
        "Peak extra inference memory after warm-up (MB)": float(peak_extra_inference_memory),

        "Serialized Model Size (MB)": float(serialized_size_mb),

        "Memory definition": "Peak process resident set size during full-test chunked inference",
        "Size definition": "Pure saved model file size on disk",
        "Execution mode": "Single-model process, full-test chunked inference",
        "Chunk size": int(CHUNK_SIZE),
        "Batch size": int(BATCH_SIZE),
        "Python": platform.python_version(),
        "CPU count": psutil.cpu_count(logical=True),
        "Total RAM (GB)": round(psutil.virtual_memory().total / (1024**3), 2),
    }

    print("\nResult:")
    for k, v in result.items():
        if isinstance(v, float):
            print(f"{k:<50}: {v:.6f}")
        else:
            print(f"{k:<50}: {v}")

    return result

def run_ae_mlp():
    import tensorflow as tf

    X = load_x_test_memmap()

    print("\nLoading AE-MLP only...")
    ae_model = tf.keras.models.load_model(find_file(AE_DIR, "ae_model_final.h5"), compile=False)
    encoder  = tf.keras.models.load_model(find_file(AE_DIR, "ae_encoder_final.h5"), compile=False)
    mlp_ae   = tf.keras.models.load_model(find_file(AE_DIR, "ae_mlp_final.h5"), compile=False)

    print("AE-MLP loaded. No other model is loaded.")

    def predict_func(X_input):
        X_pred = ae_model.predict(X_input, batch_size=BATCH_SIZE, verbose=0)
        Z = encoder.predict(X_input, batch_size=BATCH_SIZE, verbose=0)
        err = np.mean((X_input - X_pred) ** 2, axis=1, keepdims=True)
        Z_full = np.concatenate([Z, err], axis=1).astype("float32")
        y = mlp_ae.predict(Z_full, batch_size=BATCH_SIZE, verbose=0).ravel()

        del X_pred, Z, err, Z_full
        return y

    return benchmark("AE-MLP", predict_func, X)

def run_mlp():
    import tensorflow as tf

    X = load_x_test_memmap()

    print("\nLoading MLP only...")
    mlp_std = tf.keras.models.load_model(find_file(MLP_DIR, "mlp_model_final.h5"), compile=False)

    print("MLP loaded. No other model is loaded.")

    def predict_func(X_input):
        return mlp_std.predict(X_input, batch_size=BATCH_SIZE, verbose=0).ravel()

    return benchmark("MLP", predict_func, X)

def run_lgbm():
    warnings.filterwarnings("ignore", message="X does not have valid feature names.*")

    X = load_x_test_memmap()

    print("\nLoading LightGBM only...")
    lgbm = joblib.load(find_file(LGBM_DIR, "lgbm_model_final.joblib"))

    print("LightGBM loaded. No other model is loaded.")

    def predict_func(X_input):
        return lgbm.predict_proba(X_input)[:, 1]

    return benchmark("LightGBM", predict_func, X)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True, choices=["ae_mlp", "mlp", "lgbm"])
    args = parser.parse_args()

    if args.model == "ae_mlp":
        result = run_ae_mlp()
        out_name = "benchmark_ae_mlp_reviewer.csv"
    elif args.model == "mlp":
        result = run_mlp()
        out_name = "benchmark_mlp_reviewer.csv"
    elif args.model == "lgbm":
        result = run_lgbm()
        out_name = "benchmark_lgbm_reviewer.csv"

    out_csv = os.path.join(OUT_DIR, out_name)
    out_json = out_csv.replace(".csv", ".json")

    pd.DataFrame([result]).to_csv(out_csv, index=False)

    with open(out_json, "w") as f:
        json.dump(result, f, indent=2)

    print("\nSaved CSV :", out_csv)
    print("Saved JSON:", out_json)

if __name__ == "__main__":
    main()
'''

with open(SCRIPT_PATH, "w") as f:
    f.write(script_code)

print("Benchmark script created:")
print(SCRIPT_PATH)
print("\nCell 3 selesai.")

Benchmark script created:
/content/model_benchmark_reviewer/benchmark_reviewer_single_model.py

Cell 3 selesai.


In [ ]:
# ============================================================
# CELL 4 — RUN REVIEWER BENCHMARK: AE-MLP
# ============================================================

!python /content/model_benchmark_reviewer/benchmark_reviewer_single_model.py --model ae_mlp

Loading X_test as memory-mapped array...
X_test shape: (1759848, 38)
X_test dtype: float32
X_test size : 255.10 MiB

Loading AE-MLP only...
2026-06-12 00:16:49.028981: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
AE-MLP loaded. No other model is loaded.
Benchmark model : AE-MLP
Samples         : 1,759,848
Mode            : full-test chunked inference, single-model process
Chunk size      : 10,000
Batch size      : 4,096
Warm-up...
Processed: 200,000 / 1,759,848
Processed: 400,000 / 1,759,848
Processed: 600,000 / 1,759,848
Processed: 800,000 / 1,759,848
Processed: 1,000,000 / 1,759,848
Processed: 1,200,000 / 1,759,848
Processed: 1,400,000 / 1,759,848
Processed: 1,600,000 / 1,759,848

Result:
Model                                             : AE-MLP
Samples                                           : 1759848
Total inference time (s)                          : 63.690786
Latency p

In [ ]:
# ============================================================
# CELL 5 — RUN REVIEWER BENCHMARK: MLP
# ============================================================

!python /content/model_benchmark_reviewer/benchmark_reviewer_single_model.py --model mlp

Loading X_test as memory-mapped array...
X_test shape: (1759848, 38)
X_test dtype: float32
X_test size : 255.10 MiB

Loading MLP only...
2026-06-12 00:18:01.202700: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
MLP loaded. No other model is loaded.
Benchmark model : MLP
Samples         : 1,759,848
Mode            : full-test chunked inference, single-model process
Chunk size      : 10,000
Batch size      : 4,096
Warm-up...
Processed: 200,000 / 1,759,848
Processed: 400,000 / 1,759,848
Processed: 600,000 / 1,759,848
Processed: 800,000 / 1,759,848
Processed: 1,000,000 / 1,759,848
Processed: 1,200,000 / 1,759,848
Processed: 1,400,000 / 1,759,848
Processed: 1,600,000 / 1,759,848

Result:
Model                                             : MLP
Samples                                           : 1759848
Total inference time (s)                          : 38.445732
Latency per flow (ms)

In [ ]:
# ============================================================
# CELL 6 — RUN REVIEWER BENCHMARK: LIGHTGBM
# ============================================================

!python /content/model_benchmark_reviewer/benchmark_reviewer_single_model.py --model lgbm

Loading X_test as memory-mapped array...
X_test shape: (1759848, 38)
X_test dtype: float32
X_test size : 255.10 MiB

Loading LightGBM only...
LightGBM loaded. No other model is loaded.
Benchmark model : LightGBM
Samples         : 1,759,848
Mode            : full-test chunked inference, single-model process
Chunk size      : 10,000
Batch size      : 4,096
Warm-up...
Processed: 200,000 / 1,759,848
Processed: 400,000 / 1,759,848
Processed: 600,000 / 1,759,848
Processed: 800,000 / 1,759,848
Processed: 1,000,000 / 1,759,848
Processed: 1,200,000 / 1,759,848
Processed: 1,400,000 / 1,759,848
Processed: 1,600,000 / 1,759,848

Result:
Model                                             : LightGBM
Samples                                           : 1759848
Total inference time (s)                          : 273.397612
Latency per flow (ms)                             : 0.155353
Throughput (flows/s)                              : 6436.954550
CPU usage (%)                                     : 81.686

In [ ]:
# ============================================================
# CELL 7 — COMBINE REVIEWER-COMPLIANT COMPUTATIONAL TABLE
# ============================================================

import os
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/CICIDS2018/benchmark_reviewer_results"

files = [
    os.path.join(OUT_DIR, "benchmark_ae_mlp_reviewer.csv"),
    os.path.join(OUT_DIR, "benchmark_mlp_reviewer.csv"),
    os.path.join(OUT_DIR, "benchmark_lgbm_reviewer.csv"),
]

df_list = [pd.read_csv(f) for f in files]
df_final = pd.concat(df_list, ignore_index=True)

cols = [
    "Model",
    "Samples",
    "Total inference time (s)",
    "Latency per flow (ms)",
    "Throughput (flows/s)",
    "CPU usage (%)",
    "Peak Inference Memory RSS (MB)",
    "Serialized Model Size (MB)",
    "Execution mode",
]

df_paper = df_final[cols].copy()

df_paper["Total inference time (s)"] = df_paper["Total inference time (s)"].round(4)
df_paper["Latency per flow (ms)"] = df_paper["Latency per flow (ms)"].round(6)
df_paper["Throughput (flows/s)"] = df_paper["Throughput (flows/s)"].round(2)
df_paper["CPU usage (%)"] = df_paper["CPU usage (%)"].round(2)
df_paper["Peak Inference Memory RSS (MB)"] = df_paper["Peak Inference Memory RSS (MB)"].round(2)
df_paper["Serialized Model Size (MB)"] = df_paper["Serialized Model Size (MB)"].round(6)

print("=== REVIEWER-COMPLIANT COMPUTATIONAL PERFORMANCE TABLE ===")
print(df_paper.to_string(index=False))

final_csv = os.path.join(
    OUT_DIR,
    "computational_performance_reviewer_compliant.csv"
)

df_paper.to_csv(final_csv, index=False)

print("\nSaved final table to:")
print(final_csv)

=== REVIEWER-COMPLIANT COMPUTATIONAL PERFORMANCE TABLE ===
   Model  Samples  Total inference time (s)  Latency per flow (ms)  Throughput (flows/s)  CPU usage (%)  Peak Inference Memory RSS (MB)  Serialized Model Size (MB)                                    Execution mode
  AE-MLP  1759848                   63.6908               0.036191              27631.12          45.92                         1092.95                    0.286784 Single-model process, full-test chunked inference
     MLP  1759848                   38.4457               0.021846              45774.86          52.65                         1114.98                    2.409368 Single-model process, full-test chunked inference
LightGBM  1759848                  273.3976               0.155353               6436.95          81.69                          498.79                    7.348262 Single-model process, full-test chunked inference

Saved final table to:
/content/drive/MyDrive/CICIDS2018/benchmark_reviewer_results/c